In [5]:
import time
import asyncio
import httpx
async def fetch_url(client:httpx.AsyncClient,url:str,idx:int) -> dict:
    print(f"{idx}:{url}")
    resp = await client.get(url,timeout=10)
    print(f"{idx}:{resp.status_code}")
    return {"idx":idx,"status":resp.status_code,"url":url}
async def main():
    URLS = [
    "https://httpbin.org/delay/1",
    "https://httpbin.org/delay/2",
    "https://httpbin.org/delay/1",
    "https://httpbin.org/delay/1",
    "https://httpbin.org/delay/2",]

    start = time.time()
    async with httpx.AsyncClient() as client:
        tasks = [fetch_url(client,url,i) for i,url in enumerate(URLS)]
        results = await asyncio.gather(*tasks)
        elapsed = time.time() - start
        print(f"总耗时:{elapsed:.2f}秒")
        print(f"成功数{len(results)}")
        print(f"串行预计耗时:~7秒,异步并发节省了大量时间!")
if __name__ == "__main__":
    await main()



0:https://httpbin.org/delay/1
1:https://httpbin.org/delay/2
2:https://httpbin.org/delay/1
3:https://httpbin.org/delay/1
4:https://httpbin.org/delay/2
2:200
0:200
3:200
4:200
1:200
总耗时:4.01秒
成功数5
串行预计耗时:~7秒,异步并发节省了大量时间!


In [13]:
import asyncio
async def llm_stream_response(prompt:str):
    response_tokens = f"你好,你问的是{prompt}".split()
    for token in response_tokens:
        await asyncio.sleep(0.1)
        yield token
async def main():
    prompt = "什么是异步编程？"
    print(f"🤖 Prompt: {prompt}\n")
    print("📡 流式输出: ", end="", flush=True)
    full_responses = []
    async for token in llm_stream_response(prompt):
        print(token,end=" ",flush=True)
        full_responses.append(token)
    print(f"\n\n📝 完整回答: {''.join(full_responses)}")
    print(f"📊 共 {len(full_responses)} 个 token")
if __name__ == "__main__":
    await main()

🤖 Prompt: 什么是异步编程？

📡 流式输出: 你好,你问的是什么是异步编程？ 

📝 完整回答: 你好,你问的是什么是异步编程？
📊 共 1 个 token


In [15]:
import asyncio
import random
async def producer(queue:asyncio.Queue,producer_id:int,num_items:int):
    for i in range(num_items):
        item = f"数据_{producer_id}_{i}"
        await asyncio.sleep(random.uniform(0.1,0.3))
        await queue.put(item)
        print(f"  📥 生产者{producer_id} 放入: {item} (队列大小: {queue.qsize()})")
    await queue.put(None)
    print(f"  🏁 生产者{producer_id} 完成")
async def consumer(queue:asyncio.Queue,consumer_id:int):
    processed = 0
    while True:
        item = await queue.get()
        if item is None:
            await queue.put(None)
            break
        await asyncio.sleep(0.2)
        processed += 1
        print(f"  ⚙️  消费者{consumer_id} 处理: {item}")
        queue.task_done()
    print(f"  ✅ 消费者{consumer_id} 完成，共处理 {processed} 条")
async def main():
    print("=== 异步生产者-消费者模型 ===\n")
    queue = asyncio.Queue(maxsize=5)
    producers = [producer(queue,pid,4)for pid in range(2)]
    consumers = [consumer(queue,cid)for cid in range(2)]
    await asyncio.gather(*producers,*consumers)
    print("\n🎉 全部处理完成！")
if __name__ == '__main__':
    await main()

=== 异步生产者-消费者模型 ===

  📥 生产者0 放入: 数据_0_0 (队列大小: 1)
  📥 生产者1 放入: 数据_1_0 (队列大小: 2)
  📥 生产者0 放入: 数据_0_1 (队列大小: 1)
  📥 生产者1 放入: 数据_1_1 (队列大小: 2)
  ⚙️  消费者1 处理: 数据_1_0
  ⚙️  消费者0 处理: 数据_0_0
  📥 生产者0 放入: 数据_0_2 (队列大小: 1)
  ⚙️  消费者1 处理: 数据_0_1
  ⚙️  消费者0 处理: 数据_1_1
  📥 生产者1 放入: 数据_1_2 (队列大小: 1)
  📥 生产者0 放入: 数据_0_3 (队列大小: 1)
  🏁 生产者0 完成
  ⚙️  消费者1 处理: 数据_0_2
  ⚙️  消费者0 处理: 数据_1_2
  ✅ 消费者0 完成，共处理 3 条
  📥 生产者1 放入: 数据_1_3 (队列大小: 2)
  🏁 生产者1 完成
  ⚙️  消费者1 处理: 数据_0_3
  ✅ 消费者1 完成，共处理 4 条

🎉 全部处理完成！


In [ ]:
import threading
import time
def greet(name:str,delay:int):
    time.sleep(delay)
    print(f'{name}完成')
if __name__ == "__main__":
    t1 = threading.Thread(target=greet,args=('Alice',2),name='T-Alice')
    t2 = threading.Thread(target=greet,args=('Bob',1),name='T-Bob')
    t1.start()
    t2.start()
    t1.join()
    t2.join()

Bob完成
Alice完成


In [33]:
import threading
import time
def download_file(file_id:str,delay:int):
    time.sleep(delay)
if __name__ == '__main__':
    delays = [2,1,3,1,2]
    start = time.time()
    for i,d in enumerate(delays):
        download_file(i,d)
    serial_time = time.time() - start
    print('串行时间',serial_time)

    threads = []
    start = time.time()
    for i,d in enumerate(delays):
        t = threading.Thread(target=download_file,args=(i,d))
        threads.append(t)
        t.start()
    for t in threads:
        t.join()
        parallel_time = time.time() - start
    print('并行时间',parallel_time)

串行时间 9.002434492111206
并行时间 3.0012853145599365


In [34]:
import threading
import time
counter = 0
lock = threading.Lock()
def unsafe_increment(n:int):
    global counter
    for _ in range(n):
        current = counter
        time.sleep(0.0001)
        counter = current + 1
def safe_increment(n:int):
    global counter
    for _ in range(n):
        with lock:
            current = counter
            time.sleep(0.0001)
            counter = current + 1
if __name__ == '__main__':
    NUM_THREADS = 10
    INCREMENTS_PER_THREAD = 100
    EXPECTED = NUM_THREADS * INCREMENTS_PER_THREAD

    counter = 0
    threads = [threading.Thread(target=unsafe_increment,args=(INCREMENTS_PER_THREAD,)) for _ in range(NUM_THREADS)]
    for t in threads:t.start()
    for t in threads:t.join()
    print(f"❌ 不加锁: counter = {counter} (期望 {EXPECTED})")

    counter = 0
    threads = [threading.Thread(target=safe_increment,args=(INCREMENTS_PER_THREAD,)) for _ in range(NUM_THREADS)]
    for t in threads:t.start()
    for t in threads:t.join()
    print(f"✅ 加锁后: counter = {counter} (期望 {EXPECTED})")

❌ 不加锁: counter = 100 (期望 1000)
✅ 加锁后: counter = 1000 (期望 1000)
